<a href="https://colab.research.google.com/github/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/blob/main/notebooks/04_Ventaja_comparativa_revelada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Cuaderno 4. La familia de índices de Ventaja Comparativa Revelada

**Asignatura:** Inteligencia en Negocios Globales
**Semana 5 — Nivel intermedio**
**Documento base:** *Métricas de Comercio Exterior e Inteligencia de Negocios Globales. Documento 2 de 3 — Nivel intermedio: concentración, comercio intraindustrial y ventaja comparativa revelada* (Serie de recursos, 2026), sección 3.

---

## Retomamos el caso

Una empresa floricultora colombiana exporta **claveles frescos** (**HS 060312**). El Cuaderno 3 desmontó la premisa del caso: Corea del Sur no es un mercado por conquistar, es un mercado que Colombia ya domina con el 86,65 %.

Eso deja la pregunta de negocio en un lugar nuevo y más incómodo:

> Colombia vende la mitad de los claveles del mundo y controla el mercado coreano. ¿Eso es **ventaja comparativa real** —algo que Colombia hace estructuralmente mejor que los demás— o es **volumen heredado** que alguien puede disputar?

Es exactamente la pregunta que el Cuaderno 2 dejó por escrito y que ninguna métrica de concentración puede responder. La concentración mide **cuánto** y **de cuántos**; nunca mide **qué tan bueno**.

## Por qué una familia y no un índice

Esta es la familia de métricas más citada en la literatura de comercio internacional aplicada, y también la que concentra más debate metodológico. Los cinco índices se presentan **en orden cronológico**, porque cada uno nació explícitamente para corregir una limitación matemática del anterior. Entender ese linaje es entender por qué existen tantas variantes.

| Año | Índice | Qué corrige del anterior |
|---|---|---|
| 1965 | **Balassa (RCA)** | El fundacional. Mide especialización comparando con el mundo |
| 1991 | **Vollrath (RXA, RMA, RTA, RC)** | Balassa ignora las importaciones: no distingue producción eficiente de maquila |
| 1992 | **Lafay (LFI)** | Todos los anteriores comparan contra el mundo, y el ciclo macroeconómico contamina esa comparación |
| 2009 | **NRCA (Yu, Cai y Leung)** | Ninguno tiene propiedades estadísticas para usarse en regresiones sin distorsión |
| 2015 | **RSCA (Laursen)** | El rango de Balassa es asimétrico: 0 a 1 para desventaja, 1 a ∞ para ventaja |

Los presentamos en el orden del documento base, que agrupa primero las dos correcciones de escala (Balassa → Laursen) y luego las dos correcciones conceptuales (Vollrath → NRCA), cerrando con Lafay.

## Cómo usar este cuaderno

Ejecuta las celdas **en orden**. Viene configurado en modo `"github"`: descarga sus propios datos del repositorio del curso, así que **no necesitas haber ejecutado los cuadernos anteriores**.

Este cuaderno contiene una trampa numérica real —no un ejemplo de juguete— que hace que el cálculo de uno de los índices devuelva un resultado **falso sin lanzar ningún error**. Está en la sección 5. Si la lees rápido, te la pierdes.

---
# 0. Preparación del entorno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 170)

AZUL      = "#2a78d6"
ROJO      = "#e34948"
NARANJA   = "#eb6834"
VERDE     = "#3f8f6b"
GRIS_MID  = "#c3c2b7"
TINTA     = "#0b0b0b"
GRIS_TEXT = "#52514e"
GRIS_EJE  = "#898781"
REJILLA   = "#e1e0d9"

print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
# ============================================================
#  CONFIGURACION: cambia solo esta celda
# ============================================================

MODO = "github"         # opciones: "github" | "subir" | "drive" | "local"

URL_DATOS = "https://raw.githubusercontent.com/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/main/data/"

ARCHIVOS_REPO = [
    "co_exp_productos_hs2_serie.xls",
    "co_imp_productos_hs2_serie.csv",
    "mundo_exp_productos_hs2_serie.csv",
    "mundo_imp_productos_hs2_serie.csv",
    "co_exp_productos_hs2_indicadores.xls",
    "kor_060312_proveedores_serie.csv",
]

RUTA_DRIVE = "/content/drive/MyDrive/Semana 5"
RUTA_LOCAL = "."
CARPETA_SALIDA = "datos_limpios"

print(f"Modo seleccionado: {MODO}")

In [ ]:
# ============================================================
#  Ejecuta esta celda tal cual: prepara la ruta segun el modo
# ============================================================

if MODO == "github":
    import urllib.request

    RUTA_BASE = "datos_crudos"
    os.makedirs(RUTA_BASE, exist_ok=True)
    for nombre in ARCHIVOS_REPO:
        urllib.request.urlretrieve(URL_DATOS + nombre, os.path.join(RUTA_BASE, nombre))
        print(f"  descargado: {nombre}")

elif MODO == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    RUTA_BASE = RUTA_DRIVE

elif MODO == "subir":
    from google.colab import files
    print("Selecciona los seis archivos de datos.\n")
    files.upload()
    RUTA_BASE = "/content"

else:  # local
    RUTA_BASE = RUTA_LOCAL

os.makedirs(CARPETA_SALIDA, exist_ok=True)
print("\nRuta base de los datos:", RUTA_BASE)

In [ ]:
def buscar_archivo(patron, ruta_base=None):
    '''Busca un archivo por patron dentro de la ruta base, incluyendo subcarpetas.'''
    if ruta_base is None:
        ruta_base = RUTA_BASE
    coincidencias = sorted(glob.glob(os.path.join(ruta_base, "**", patron), recursive=True))
    if len(coincidencias) == 0:
        raise FileNotFoundError(f"No encontre ningun archivo que coincida con '{patron}'.")
    return coincidencias[0]


def a_numero(serie):
    '''Convierte a numero una columna de Trade Map que viene como texto con comas.'''
    return pd.to_numeric(
        pd.Series(serie).astype(str)
                        .str.replace(",", "", regex=False)
                        .str.replace(r"[^0-9.\-]", "", regex=True)
                        .replace("", np.nan),
        errors="coerce",
    )


def limpiar_codigo(serie):
    '''Quita el apostrofe con que Trade Map fuerza los codigos a texto.'''
    return serie.astype(str).str.strip().str.lstrip("'").str.strip()


ANIOS = [str(a) for a in range(2021, 2026)]


def leer_canasta(ruta):
    '''Lee la canasta por capitulo HS, venga como CSV o como .xls que en realidad es HTML.'''
    with open(ruta, encoding="utf-8", errors="ignore") as f:
        primeras_letras = f.read(200).lstrip().lower()

    if primeras_letras.startswith("<"):
        tabla = max(pd.read_html(ruta), key=lambda t: t.shape[0])
    else:
        tabla = pd.read_csv(ruta)

    tabla = tabla.iloc[:, -7:]
    tabla.columns = ["codigo", "producto"] + ANIOS
    tabla["codigo"] = limpiar_codigo(tabla["codigo"])
    for anio in ANIOS:
        tabla[anio] = a_numero(tabla[anio])

    return tabla.dropna(subset=["2025"]).reset_index(drop=True)


ARCHIVOS = {
    "co_exp":     buscar_archivo("co_exp_productos_hs2_serie.xls"),
    "co_imp":     buscar_archivo("co_imp_productos_hs2_serie.csv"),
    "mundo_exp":  buscar_archivo("mundo_exp_productos_hs2_serie.csv"),
    "mundo_imp":  buscar_archivo("mundo_imp_productos_hs2_serie.csv"),
    "co_exp_ind": buscar_archivo("co_exp_productos_hs2_indicadores.xls"),
    "kor_prov":   buscar_archivo("kor_060312_proveedores_serie.csv"),
}
for nombre, ruta in ARCHIVOS.items():
    print(f"{nombre:12s} -> {os.path.basename(ruta)}")

## 0.1. La tabla maestra

Toda la familia VCR se calcula sobre **cuatro cantidades** por capítulo: lo que Colombia exporta e importa, y lo que el mundo exporta e importa. Las juntamos en una sola tabla, y a partir de ahí lo único que cambia entre índice e índice es la transformación matemática que se les aplica.

Ese es literalmente el punto del documento base: *"un mismo archivo descargado de Trade Map puede alimentar las ocho métricas"*.

In [ ]:
co_exp    = leer_canasta(ARCHIVOS["co_exp"])
co_imp    = leer_canasta(ARCHIVOS["co_imp"])
mundo_exp = leer_canasta(ARCHIVOS["mundo_exp"])
mundo_imp = leer_canasta(ARCHIVOS["mundo_imp"])

maestra = (
    co_exp[["codigo", "producto", "2025"]].rename(columns={"2025": "X_col"})
      .merge(co_imp[["codigo", "2025"]].rename(columns={"2025": "M_col"}),   on="codigo")
      .merge(mundo_exp[["codigo", "2025"]].rename(columns={"2025": "X_mun"}), on="codigo")
      .merge(mundo_imp[["codigo", "2025"]].rename(columns={"2025": "M_mun"}), on="codigo")
)

# ¡IMPORTANTE! Forzamos punto flotante. La razon esta en la seccion 5,
# y no es una manía de estilo: sin esto, uno de los indices sale mal.
for columna in ["X_col", "M_col", "X_mun", "M_mun"]:
    maestra[columna] = maestra[columna].astype(float)

TOTALES  = maestra[maestra["codigo"] == "TOTAL"].iloc[0]
capitulos = maestra[maestra["codigo"] != "TOTAL"].copy().reset_index(drop=True)

print(f"Capitulos en la tabla maestra: {len(capitulos)}\n")
print("Los cuatro agregados de referencia (miles de USD, 2025):\n")
print(f"  X_col  Colombia exporta : {TOTALES['X_col']:>18,.0f}")
print(f"  M_col  Colombia importa : {TOTALES['M_col']:>18,.0f}")
print(f"  X_mun  Mundo exporta    : {TOTALES['X_mun']:>18,.0f}")
print(f"  M_mun  Mundo importa    : {TOTALES['M_mun']:>18,.0f}")
print(f"\n  Colombia pesa el {TOTALES['X_col'] / TOTALES['X_mun'] * 100:.3f} % de las exportaciones mundiales.")
print("\nPrimeras filas:\n")
print(capitulos.head(4).assign(producto=lambda d: d["producto"].str[:38]).to_string(index=False))

---
# 1. Métrica 7 — El índice fundacional de Balassa (RCA)

## 1.1. Qué es

Formalizado por Béla Balassa en 1965, mide el grado de **especialización** de un país en un producto, comparando qué tan importante es ese producto dentro de la canasta exportadora del país frente a qué tan importante es ese mismo producto dentro del comercio mundial total.

## 1.2. Para qué sirve y cómo se usa

Es el punto de partida obligado de cualquier análisis de competitividad sectorial. Responde a una pregunta simple: *"¿mi país exporta proporcionalmente MÁS de este producto que el resto del mundo, o menos?"*.

Se usa como **primer filtro**: identifica en qué productos vale la pena profundizar con métodos más rigurosos y costosos (NRCA, Vollrath, modelos gravitacionales) y en cuáles claramente no hay ninguna señal de especialización que perseguir.

## 1.3. La fórmula

$$IVCR^k_i = \frac{X^k_i / X^T_i}{X^k_w / X^T_w}$$

## 1.4. Explicación matemática detallada

El **numerador** $(X^k_i / X^T_i)$ es la participación del producto $k$ dentro de las exportaciones totales del país $i$. Si el café representa el 15 % de todo lo que exporta un país, vale 0,15.

El **denominador** $(X^k_w / X^T_w)$ es la participación de ese mismo producto dentro del comercio mundial. Si el café es el 0,5 % del comercio mundial de bienes, vale 0,005.

Dividir el numerador entre el denominador responde: *"¿cuántas veces más —o menos— importante es este producto para mi país, comparado con su importancia para el mundo?"*.

Si el resultado es mayor que 1, el país exporta proporcionalmente más de ese bien que el promedio mundial. Está **revelando**, a través de su comportamiento comercial observado y no de un supuesto teórico, una especialización en ese producto. De ahí el nombre.

## 1.5. Ejemplo numérico paso a paso

In [ ]:
def rca_balassa(x_pais, x_pais_total, x_mundo, x_mundo_total):
    '''Indice de Ventaja Comparativa Revelada de Balassa (1965).

    Compara el peso del producto en la canasta del pais contra su peso
    en el comercio mundial. Mayor que 1 = especializacion revelada.
    '''
    participacion_pais  = np.asarray(x_pais, dtype=float)  / float(x_pais_total)
    participacion_mundo = np.asarray(x_mundo, dtype=float) / float(x_mundo_total)

    return participacion_pais / participacion_mundo

In [ ]:
print("Los dos ejemplos del documento base:\n")

# El cafe: 15 % de las exportaciones del pais, 0,5 % del comercio mundial
rca_cafe = rca_balassa(0.15, 1, 0.005, 1)
print(f"  Cafe        15 % del pais / 0,5 % del mundo   ->  RCA = {rca_cafe:.1f}")
print( "              el cafe es 30 veces mas importante para este pais que para el mundo\n")

# La maquinaria: 0,2 % de las exportaciones del pais, 8 % del comercio mundial
rca_maq = rca_balassa(0.002, 1, 0.08, 1)
print(f"  Maquinaria  0,2 % del pais / 8 % del mundo    ->  RCA = {rca_maq:.3f}")
print( "              desventaja comparativa profunda: el pais practicamente no participa")

Fíjate en los dos números: **30** y **0,025**. Están describiendo situaciones simétricas —especialización extrema en un caso, ausencia extrema en el otro— pero los números no son simétricos en absoluto. Uno se fue a 30 y podría haberse ido a 300; el otro está aplastado contra el cero.

Esa asimetría es el problema que va a ocupar la sección 2, y es la razón por la que existe el resto de la familia.

## 1.6. Cálculo sobre nuestros datos reales

In [ ]:
capitulos["RCA"] = rca_balassa(
    capitulos["X_col"], TOTALES["X_col"],
    capitulos["X_mun"], TOTALES["X_mun"],
)

con_ventaja = capitulos[capitulos["RCA"] > 1]

print(f"Capitulos con ventaja comparativa revelada (RCA > 1): {len(con_ventaja)} de {len(capitulos)}\n")
print("Los diez mas especializados:\n")
tabla = capitulos.nlargest(10, "RCA")[["codigo", "producto", "X_col", "RCA"]].copy()
tabla["producto"] = tabla["producto"].str[:46]
print(tabla.to_string(index=False))

In [ ]:
# El producto del caso, en su contexto
flores = capitulos[capitulos["codigo"] == "06"].iloc[0]

print("HS06 - Plantas vivas y productos de la floricultura\n")
print(f"  Colombia exporta       : {flores['X_col']:>14,.0f} miles USD")
print(f"  Participacion en su canasta : {flores['X_col'] / TOTALES['X_col'] * 100:>9.4f} %")
print(f"  El mundo exporta       : {flores['X_mun']:>14,.0f} miles USD")
print(f"  Participacion mundial  : {flores['X_mun'] / TOTALES['X_mun'] * 100:>14.4f} %")
print(f"\n  RCA = {flores['X_col'] / TOTALES['X_col'] * 100:.4f} / {flores['X_mun'] / TOTALES['X_mun'] * 100:.4f} = {flores['RCA']:.2f}")
print(f"\n  Las flores son {flores['RCA']:.0f} veces mas importantes para la canasta colombiana")
print( "  que para el comercio mundial promedio. Es la especializacion mas alta del pais.")

In [ ]:
# ¿Sube o baja en el tiempo? Recalculamos el RCA para los cinco anios
serie_rca = co_exp[["codigo", "producto"]].copy()
for anio in ANIOS:
    total_col = co_exp.loc[co_exp["codigo"] == "TOTAL", anio].iloc[0]
    total_mun = mundo_exp.loc[mundo_exp["codigo"] == "TOTAL", anio].iloc[0]
    mundo_anio = mundo_exp.set_index("codigo")[anio]
    serie_rca[anio] = rca_balassa(
        co_exp[anio].values, total_col,
        co_exp["codigo"].map(mundo_anio).values, total_mun,
    )

serie_rca = serie_rca[serie_rca["codigo"] != "TOTAL"]
print("Trayectoria del RCA en los capitulos donde Colombia esta mas especializada:\n")
print(serie_rca[serie_rca["codigo"].isin(["06", "09", "08", "27", "17"])]
      .assign(producto=lambda d: d["producto"].str[:34]).to_string(index=False))

In [ ]:
datos = capitulos.nlargest(12, "RCA").sort_values("RCA")
colores = [NARANJA if c == "06" else AZUL for c in datos["codigo"]]
etiquetas = [f"HS{c} · {p[:30]}" for c, p in zip(datos["codigo"], datos["producto"])]

fig, ax = plt.subplots(figsize=(9.5, 6))
barras = ax.barh(etiquetas, datos["RCA"], color=colores, height=0.66)

for barra, valor in zip(barras, datos["RCA"]):
    ax.text(valor + 0.7, barra.get_y() + barra.get_height() / 2,
            f"{valor:.1f}", va="center", fontsize=9, color=GRIS_TEXT)

ax.axvline(1, color=GRIS_MID, linewidth=1.4)
ax.text(1.4, -0.9, "RCA = 1: el umbral de la ventaja", fontsize=8.5, color=GRIS_EJE)

ax.set_xlabel("Índice de Balassa (RCA)   ·   escala sin techo", fontsize=10, color=GRIS_TEXT)
ax.set_title("Colombia está más especializada en flores que en ningún otro capítulo\n"
             "Ventaja comparativa revelada, doce capítulos más altos, 2025",
             fontsize=13, color=TINTA, loc="left", pad=14)
ax.xaxis.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=8.5)

plt.figtext(0.01, -0.04,
            "En naranja: HS06, el capitulo del caso.\n"
            "Fuente: elaboracion propia con datos de Trade Map (ITC, 2025).",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

## 1.7. Interpretación y umbrales

$IVCR > 1$ certifica ventaja comparativa revelada (especialización); $0 \le IVCR < 1$ denota desventaja.

**No existe límite superior matemático.** Puede llegar a cientos en productos hiperespecializados de economías pequeñas. Y eso, que parece un detalle técnico, es precisamente el origen de su principal limitación.

## 1.8. Impacto en el negocio y la decisión que habilita

El propio Balassa (1965) advirtió que el índice **ignora las importaciones**: un país puede tener RCA alto porque produce eficientemente, o porque simplemente reexporta con poco valor agregado insumos importados. La sección 3 corrige exactamente eso.

French (2017), en un análisis que evalúa "para qué sirve realmente" el RCA usando modelos cuantitativos de comercio, concluye que los índices RCA convencionales **no son consistentes** con la noción teórica de ventaja comparativa, pero sí son útiles para tres tareas concretas de negocio:

1. Identificar el patrón fundamental de especialización de un país.
2. Evaluar el efecto diferencial de cambios arancelarios sobre distintos productores.
3. Identificar qué países son competidores cercanos en un mercado específico.

Es decir: el RCA de Balassa es una herramienta de **diagnóstico rápido y comparación de competidores**, no un instrumento de precisión para decisiones de inversión de alto riesgo.

## 1.9. De dónde se saca exactamente el dato

Las cuatro cantidades de la fórmula se obtienen de Trade Map (ITC, 2025), filtrando por código HS del producto, país reportante y "Mundo" como socio de referencia. La limitación de asimetría estadística está documentada en Liu et al. (2019), Sałamaga (2015) y de Benedictis y Tamberi (2001).

---
# 2. Métrica 8 — La corrección de Laursen: RSCA

## 2.1. Qué es

Transformación del índice de Balassa propuesta por Laursen (2015) para resolver su problema de **asimetría estadística**, reescalándolo a un rango simétrico centrado en cero.

## 2.2. Para qué sirve y cómo se usa

Se usa en los mismos contextos que el RCA, pero cuando el analista necesita usar el resultado como **variable de entrada en un modelo estadístico** —regresión, correlación, prueba de hipótesis— donde la asimetría del RCA original produciría estimaciones sesgadas.

## 2.3. La fórmula

$$RSCA = \frac{RCA - 1}{RCA + 1}$$

## 2.4. Explicación matemática detallada

Esta transformación "aplana" cualquier valor entre 0 e infinito hacia el rango $[-1, 1]$:

- Cuando $RCA = 1$ (punto neutro), el numerador se anula y $RSCA = 0$.
- Cuando $RCA \to \infty$ (especialización extrema), $RSCA \to 1$.
- Cuando $RCA \to 0$ (sin ninguna exportación del bien), $RSCA \to -1$.

En el RCA original, el "espacio" entre 0 y 1 —toda la desventaja— está mucho más comprimido que el espacio entre 1 e infinito. El RSCA distribuye el rango de forma simétrica.

## 2.5. Ejemplo numérico paso a paso

In [ ]:
def rsca_laursen(rca):
    '''Ventaja Comparativa Revelada Simetrica (Laursen, 2015).

    Reescala el RCA de Balassa al rango [-1, 1], con 0 como punto neutro exacto.
    '''
    rca = np.asarray(rca, dtype=float)
    return (rca - 1) / (rca + 1)

In [ ]:
print("Los mismos dos ejemplos, ahora simetricos:\n")
print(f"  Cafe        RCA = {rca_cafe:6.3f}   ->  RSCA = {float(rsca_laursen(rca_cafe)):+.3f}")
print(f"  Maquinaria  RCA = {rca_maq:6.3f}   ->  RSCA = {float(rsca_laursen(rca_maq)):+.3f}")
print()
print("  Ahora ambos estan en el mismo rango acotado [-1, 1], lo que permite")
print("  promediarlos, graficarlos juntos o meterlos en una regresion sin")
print("  violar los supuestos estadisticos habituales. Con 30 y 0,025 no se podia.")

In [ ]:
capitulos["RSCA"] = rsca_laursen(capitulos["RCA"])

print(f"Rango del RCA  : de {capitulos['RCA'].min():.4f} a {capitulos['RCA'].max():.2f}")
print(f"Rango del RSCA : de {capitulos['RSCA'].min():+.4f} a {capitulos['RSCA'].max():+.4f}\n")

print("El capitulo del caso:")
f = capitulos[capitulos["codigo"] == "06"].iloc[0]
print(f"  HS06   RCA = {f['RCA']:.2f}   ->   RSCA = {f['RSCA']:+.4f}")

## 2.6. La prueba de que la corrección funciona

Un índice sirve para una regresión cuando su distribución no está deformada. Comparemos las dos distribuciones de forma numérica, sin depender de la impresión visual.

In [ ]:
print("Como se distribuyen los 97 capitulos segun cada indice:\n")
resumen = pd.DataFrame({
    "RCA":  capitulos["RCA"].describe(),
    "RSCA": capitulos["RSCA"].describe(),
})
print(resumen.to_string())

print(f"\n  Asimetria (skewness) del RCA  : {capitulos['RCA'].skew():>8.3f}")
print(f"  Asimetria (skewness) del RSCA : {capitulos['RSCA'].skew():>8.3f}")
print("\n  Una distribucion simetrica tiene asimetria cero. El RCA se dispara porque")
print("  unos pocos capitulos con valores enormes arrastran toda la cola derecha.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

ax1.hist(capitulos["RCA"], bins=40, color=AZUL, edgecolor="white", linewidth=0.6)
ax1.axvline(1, color=NARANJA, linewidth=1.6)
ax1.set_title("RCA de Balassa: cola derecha infinita", fontsize=11.5, color=TINTA, loc="left", pad=10)
ax1.set_xlabel("RCA   (umbral en 1)", fontsize=9.5, color=GRIS_TEXT)

ax2.hist(capitulos["RSCA"], bins=40, color=VERDE, edgecolor="white", linewidth=0.6)
ax2.axvline(0, color=NARANJA, linewidth=1.6)
ax2.set_title("RSCA de Laursen: acotado y simétrico", fontsize=11.5, color=TINTA, loc="left", pad=10)
ax2.set_xlabel("RSCA   (umbral en 0)", fontsize=9.5, color=GRIS_TEXT)

for ax in (ax1, ax2):
    ax.set_ylabel("Número de capítulos", fontsize=9.5, color=GRIS_TEXT)
    ax.yaxis.grid(True, color=REJILLA, linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ["top", "right", "left"]:
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color(GRIS_EJE)
    ax.tick_params(colors=GRIS_EJE, labelsize=9)

plt.figtext(0.01, -0.04,
            "Los mismos 97 capitulos, las mismas exportaciones. Solo cambia la escala.\n"
            "Fuente: elaboracion propia con datos de Trade Map (ITC, 2025).",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

In [ ]:
# Una comprobacion que sorprende: el RSCA NO cambia el orden de los capitulos
orden_rca  = capitulos.sort_values("RCA",  ascending=False)["codigo"].tolist()
orden_rsca = capitulos.sort_values("RSCA", ascending=False)["codigo"].tolist()

print(f"¿El RSCA reordena los capitulos respecto al RCA?  {'NO' if orden_rca == orden_rsca else 'SI'}")
print(f"Correlacion de rangos (Spearman): {capitulos['RCA'].corr(capitulos['RSCA'], method='spearman'):.6f}")
print()
print("  La transformacion de Laursen es monotona creciente: si A tenia mas RCA que B,")
print("  tambien tendra mas RSCA. No corrige QUE producto esta mejor posicionado,")
print("  corrige la ESCALA en que se mide esa posicion. Es una correccion estadistica,")
print("  no analitica: sirve para modelar, no para cambiar el diagnostico.")

## 2.7. Interpretación y umbrales

Rango $[-1, 1]$, con **0 como punto neutro exacto**. Laursen (2015) comparó la RSCA con otras alternativas —índice de Michaely, contribución a la balanza comercial, chi-cuadrado, índice de comercio neto de Bowen— y concluyó que, en balance, la RSCA es **la mejor medida disponible de ventaja comparativa para fines econométricos**, por su simplicidad de cálculo y sus buenas propiedades estadísticas.

## 2.8. Impacto en el negocio y la decisión que habilita

Para un investigador que necesita construir un modelo predictivo, usar RSCA en lugar de RCA evita conclusiones espurias derivadas de la distribución no normal del índice original.

Para uso puramente descriptivo —un reporte gerencial, una presentación a un comité— el RCA de Balassa sigue siendo **más intuitivo de comunicar**. "43 veces más especializado que el promedio mundial" se explica solo; un RSCA de 0,955 no le dice nada a nadie.

La regla práctica: **RCA para comunicar, RSCA para modelar.**

## 2.9. De dónde se saca exactamente el dato

Se calcula sobre el mismo insumo de Trade Map usado para el RCA de Balassa. No requiere datos adicionales.

---
# 3. Métrica 9 — La corrección de Vollrath: integrar oferta y demanda

## 3.1. Qué es

Vollrath (1991) propuso una familia de tres índices —Ventaja Relativa de Exportación (**RXA**), Ventaja Relativa de Importación (**RMA**) y Ventaja Comercial Relativa (**RTA**)— culminando en el Índice de **Competitividad Revelada (RC)**, que corrige la principal debilidad del RCA de Balassa: **ignorar por completo las importaciones**.

## 3.2. Para qué sirve y cómo se usa

Se usa cuando existe sospecha de que la alta exportación de un producto no refleja producción eficiente sino **ensamblaje de partes importadas** (maquila), un fenómeno muy común en electrónica, textiles y autopartes.

El RC obliga a mirar simultáneamente cuánto exporta e importa el país de ese mismo bien antes de certificar una ventaja competitiva real.

## 3.3. Las fórmulas

$$RXA^k_i = \frac{X^k_i / X^{T-k}_i}{X^k_{w-i} / X^{T-k}_{w-i}} \qquad RMA^k_i = \frac{M^k_i / M^{T-k}_i}{M^k_{w-i} / M^{T-k}_{w-i}}$$

$$RTA = RXA - RMA \qquad\qquad RC = \ln(RXA) - \ln(RMA)$$

## 3.4. Explicación matemática detallada

**RXA** se calcula casi igual que el RCA de Balassa, con una diferencia importante: ajusta los denominadores para **excluir el propio país y el propio producto** de los agregados. Los superíndices $T-k$ y $w-i$ significan exactamente eso: "todos los productos menos $k$" y "todo el mundo menos el país $i$".

¿Por qué importa? Porque si Colombia representa el 50 % del comercio mundial de claveles, incluirse a sí misma en el denominador es compararse consigo misma. El ajuste elimina ese doble conteo. En productos donde el país es marginal la diferencia es despreciable; en productos donde domina —justamente los que nos interesan— es grande.

**RMA** es lo mismo pero sobre importaciones: mide qué tan importante es ese producto dentro de las importaciones del país, comparado con su peso en las importaciones mundiales.

**RC** resta los logaritmos naturales de ambos. Restar logaritmos, en lugar de restar los valores directamente, tiene dos efectos deseados:

1. Convierte una razón $(RXA/RMA)$ en una diferencia, matemáticamente más manejable.
2. Fuerza el resultado a ser **simétrico alrededor de cero**, igual que hacía la transformación de Laursen sobre el RCA.

Si $RXA > RMA$ —el país exporta relativamente más de lo que importa de ese bien— entonces $RC > 0$: ventaja neta. Si es al revés, $RC < 0$: el país depende de importar ese bien más de lo que lo exporta en términos relativos, lo que sugiere que su aparente fortaleza exportadora podría depender de insumos externos.

## 3.5. Ejemplo numérico paso a paso

In [ ]:
def indices_vollrath(x_pais, m_pais, x_mundo, m_mundo, totales):
    '''Familia de indices de Vollrath (1991): RXA, RMA, RTA y RC.

    A diferencia del RCA de Balassa, los denominadores excluyen el propio pais
    y el propio producto, para evitar el doble conteo que distorsiona el indice
    justo en los productos donde el pais es grande.
    '''
    x_pais  = np.asarray(x_pais,  dtype=float)
    m_pais  = np.asarray(m_pais,  dtype=float)
    x_mundo = np.asarray(x_mundo, dtype=float)
    m_mundo = np.asarray(m_mundo, dtype=float)

    xt_pais, mt_pais = float(totales["X_col"]), float(totales["M_col"])
    xt_mundo, mt_mundo = float(totales["X_mun"]), float(totales["M_mun"])

    # Ventaja Relativa de Exportacion
    resto_pais_x  = xt_pais - x_pais                               # el pais, sin este producto
    resto_mundo_x = x_mundo - x_pais                               # el mundo, sin este pais
    resto_ambos_x = (xt_mundo - x_mundo) - (xt_pais - x_pais)      # todo lo demas
    rxa = (x_pais / resto_pais_x) / (resto_mundo_x / resto_ambos_x)

    # Ventaja Relativa de Importacion, simetrica
    resto_pais_m  = mt_pais - m_pais
    resto_mundo_m = m_mundo - m_pais
    resto_ambos_m = (mt_mundo - m_mundo) - (mt_pais - m_pais)
    rma = (m_pais / resto_pais_m) / (resto_mundo_m / resto_ambos_m)

    return pd.DataFrame({
        "RXA": rxa,
        "RMA": rma,
        "RTA": rxa - rma,
        "RC":  np.log(rxa) - np.log(rma),
    })

In [ ]:
print("El ejemplo del documento base: electronica de consumo\n")

rxa_ejemplo, rma_ejemplo = 8.0, 6.0
rc_ejemplo = np.log(rxa_ejemplo) - np.log(rma_ejemplo)

print(f"  RXA = {rxa_ejemplo:.0f}   fuerte ventaja exportadora aparente")
print(f"  RMA = {rma_ejemplo:.0f}   pero tambien importa masivamente componentes")
print(f"\n  RC  = ln({rxa_ejemplo:.0f}) - ln({rma_ejemplo:.0f}) = {np.log(rxa_ejemplo):.3f} - {np.log(rma_ejemplo):.3f} = {rc_ejemplo:.3f}")
print()
print("  El RC sigue siendo positivo (hay ventaja neta) pero mucho mas moderado")
print("  que lo que sugeriria el RXA aislado. La distancia entre RXA=8 y RC=0,29 es")
print("  la senal de que buena parte de esa 'ventaja' depende de insumos importados:")
print("  tipico de una industria de ensamblaje, no de manufactura integrada.")

## 3.6. Cálculo sobre nuestros datos reales

In [ ]:
vollrath = indices_vollrath(
    capitulos["X_col"], capitulos["M_col"],
    capitulos["X_mun"], capitulos["M_mun"],
    TOTALES,
)
capitulos = pd.concat([capitulos, vollrath], axis=1)

indefinidos = capitulos["RC"].isna().sum()
print(f"Capitulos con RC indefinido (algun flujo en cero): {indefinidos} de {len(capitulos)}\n")
print("El indice se vuelve indefinido cuando el pais tiene cero exportaciones o cero")
print("importaciones del bien: el logaritmo de cero no existe. Es una limitacion")
print("practica real en economias con matrices comerciales dispersas. Aqui no ocurre")
print("porque Colombia comercia en los 97 capitulos, pero no hay que darlo por hecho.\n")

print("El capitulo del caso:\n")
f = capitulos[capitulos["codigo"] == "06"].iloc[0]
print(f"  HS06   RXA = {f['RXA']:7.2f}   (ventaja exportadora)")
print(f"         RMA = {f['RMA']:7.2f}   (Colombia casi no importa flores)")
print(f"         RTA = {f['RTA']:+7.2f}")
print(f"         RC  = {f['RC']:+7.4f}")
print(f"\n  Comparalo con el RCA de Balassa, que era {f['RCA']:.2f}.")
print( "  El RXA es MAYOR que el RCA porque al excluir a Colombia del denominador")
print( "  mundial desaparece el efecto de compararse consigo misma: Colombia es la")
print( "  mitad del mercado mundial de claveles, asi que ese ajuste no es menor.")

## 3.7. Los cuatro capítulos donde Vollrath desmiente a Balassa

Aquí es donde el índice gana su sueldo. Si Balassa y Vollrath siempre coincidieran, no haría falta calcular los dos.

In [ ]:
desmentidos = capitulos[(capitulos["RCA"] > 1) & (capitulos["RC"] < 0)]

print("Capitulos con ventaja segun Balassa (RCA > 1) pero SIN ventaja segun Vollrath (RC < 0):\n")
tabla = desmentidos[["codigo", "producto", "X_col", "M_col", "RCA", "RXA", "RMA", "RC"]].copy()
tabla["producto"] = tabla["producto"].str[:38]
print(tabla.to_string(index=False))

print(f"\n  Son {len(desmentidos)} de los {int((capitulos['RCA'] > 1).sum())} capitulos que Balassa daba por ventajosos.")
print()
print("  En los cuatro, Colombia importa mucho de lo mismo que exporta. El RCA no lo")
print("  ve porque solo mira exportaciones; el RC lo detecta de inmediato. Es la firma")
print("  estadistica de una industria que transforma insumos importados mas que de una")
print("  que produce con recursos propios.")

In [ ]:
datos = capitulos.nlargest(14, "X_col").sort_values("RC")
etiquetas = [f"HS{c} · {p[:26]}" for c, p in zip(datos["codigo"], datos["producto"])]
colores = [NARANJA if c == "06" else (AZUL if v >= 0 else ROJO) for c, v in zip(datos["codigo"], datos["RC"])]

fig, ax = plt.subplots(figsize=(9.5, 6))
barras = ax.barh(etiquetas, datos["RC"], color=colores, height=0.66)

for barra, valor in zip(barras, datos["RC"]):
    desplazamiento = 0.12 if valor >= 0 else -0.12
    alineacion = "left" if valor >= 0 else "right"
    ax.text(valor + desplazamiento, barra.get_y() + barra.get_height() / 2,
            f"{valor:+.2f}", va="center", ha=alineacion, fontsize=9, color=GRIS_TEXT)

ax.axvline(0, color=GRIS_MID, linewidth=1.4)
ax.set_xlabel("Índice de Competitividad Revelada (RC)   ·   simétrico alrededor de cero",
              fontsize=10, color=GRIS_TEXT)
ax.set_title("Vollrath descuenta lo que un país importa de lo mismo que exporta\n"
             "RC de los 14 capítulos con más exportación colombiana, 2025",
             fontsize=13, color=TINTA, loc="left", pad=14)
ax.set_xlim(datos["RC"].min() - 1.1, datos["RC"].max() + 1.1)
ax.xaxis.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=8.5)

plt.figtext(0.01, -0.04,
            "En naranja HS06. En rojo, competitividad revelada negativa.\n"
            "Fuente: elaboracion propia con datos de Trade Map (ITC, 2025).",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

## 3.8. Interpretación y umbrales

$RC > 0$ confirma **ventaja competitiva neta y global**, ponderando simultáneamente oferta (exportaciones relativas) y demanda (importaciones relativas). $RC < 0$ indica que el país es, en términos relativos, más importador que exportador de ese bien.

El índice se vuelve **matemáticamente indefinido** cuando el país tiene cero exportaciones o cero importaciones del bien. Es una limitación práctica real al aplicarlo a productos muy nuevos o muy nicho.

## 3.9. Impacto en el negocio y la decisión que habilita

Para un analista de debida diligencia (*due diligence*) que evalúa si adquirir o invertir en una empresa exportadora, el RC es una **prueba de resistencia**: si el RC de la industria es bajo o negativo pese a que las cifras de exportación de la empresa se ven bien, existe riesgo de que el modelo de negocio dependa de una cadena de suministro importada vulnerable a devaluaciones, aranceles o rupturas logísticas.

Ese es el tipo de riesgo que **no aparece en un RCA de Balassa simple** pero sí en el RC de Vollrath.

Para la empresa del caso, el resultado es tranquilizador: con RMA ≈ 1,04, Colombia no depende de importar flores para exportarlas. Su ventaja no es de ensamblaje.

El índice se sigue usando activamente entre 2021 y 2025 para frutas (melocotón turco: Bayav y Engindeniz, 2021), carne halal (Mohd Yusoff et al., 2022), electrónicos ASEAN (Maqbool et al., 2021) y agricultura comparada entre doce países (Gül et al., 2024).

## 3.10. De dónde se saca exactamente el dato

Exportaciones e importaciones bilaterales y mundiales del producto, de Trade Map (ITC, 2025). Danna-Buitrago y Gnabo (2021) proponen además una nueva clase de índices RCA derivados críticamente de Vollrath (1991), que preservan simetría y evitan sesgo por tamaño de país.

---
# 4. Métrica 10 — El NRCA de Yu, Cai y Leung: la simetría definitiva

## 4.1. Qué es

Publicado por Yu, Cai y Leung en 2009, el Índice Normalizado de Ventaja Comparativa Revelada (**NRCA**) abandona por completo la estructura de **cociente** de Balassa. En su lugar mide, en valor absoluto, la **distancia** entre lo que un país realmente exporta de un bien y lo que exportaría si tuviera exactamente el peso "neutral" que le correspondería según su tamaño en el comercio mundial.

## 4.2. Para qué sirve y cómo se usa

Es, a 2026, **el índice más usado en la investigación académica reciente** sobre competitividad exportadora, precisamente porque es el único de la familia con propiedades matemáticas suficientemente robustas para usarse en regresiones econométricas de panel y comparaciones entre países sin distorsión estadística.

Se usa cuando el objetivo no es solo describir sino **modelar**.

## 4.3. La fórmula

$$NRCA^k_i = \frac{X^k_i}{X^t_w} - \frac{X^T_i \cdot X^k_w}{(X^t_w)^2}$$

## 4.4. Explicación matemática detallada

El **primer término** $(X^k_i / X^t_w)$ es la participación real y observada del producto $k$ exportado por el país $i$ dentro del comercio mundial total.

El **segundo término** es el punto de "ventaja comparativa neutral": la participación que *debería* tener el país $i$ en el producto $k$ si exportara ese bien exactamente en la misma proporción que su peso general en el comercio mundial. Se obtiene multiplicando el peso del país $(X^T_i / X^t_w)$ por el peso del producto $(X^k_w / X^t_w)$.

La resta entre lo observado y lo neutral es la **desviación real**. Si el país exporta más de lo que su tamaño neutral predeciría, el NRCA es positivo; si exporta menos, negativo.

Esta construcción —**restar en lugar de dividir**— es la clave de su simetría. No hay ninguna operación que genere valores indefinidos: no se divide por el valor del país, así que no hay división por cero, y no hay logaritmos.

Y tiene una propiedad **aditiva** elegante: si se suma el NRCA de un país en todos sus productos, o el NRCA de todos los países en un mismo producto, el resultado da **exactamente cero**. Es una propiedad de conservación que ningún otro índice de esta familia tiene, y que resulta muy útil para verificar que un cálculo está bien hecho.

Vamos a usarla. Y va a salvarnos.

## 4.5. La trampa: el desbordamiento de enteros

Mira otra vez el denominador de la fórmula: $(X^t_w)^2$. El comercio mundial total de 2025 fue de unos **25.609 millones de miles de dólares**. Elevado al cuadrado, eso es aproximadamente $6{,}56 \times 10^{20}$.

El tipo de dato con que pandas guarda por defecto los números enteros leídos de un archivo es `int64`, y su valor máximo es $9{,}22 \times 10^{18}$.

$6{,}56 \times 10^{20}$ **no cabe**. Y cuando no cabe, NumPy no lanza un error: da la vuelta al contador y devuelve un número equivocado, a veces negativo. El cálculo continúa, produce resultados de aspecto razonable, y todo lo que viene después está mal.

In [ ]:
comercio_mundial = np.int64(TOTALES["X_mun"])

print("Elevar al cuadrado el comercio mundial total\n")
print(f"  X_tw                     = {comercio_mundial:,}")
print(f"  Maximo que admite int64  = {np.iinfo(np.int64).max:,}")
print()

with np.errstate(over="ignore"):
    cuadrado_entero = comercio_mundial * comercio_mundial

print(f"  X_tw^2 en int64          = {cuadrado_entero:,}      <-- NEGATIVO: imposible")
print(f"  X_tw^2 en float          = {float(comercio_mundial) ** 2:.6e}      <-- correcto")
print()
print("  NumPy no aviso de nada. Simplemente devolvio un numero equivocado.")

In [ ]:
# Veamos que pasa si calculamos el NRCA sin cuidar el tipo de dato.
# Reconstruimos las columnas como enteros a proposito, que es como llegarian
# de un read_csv sin conversion explicita.

x_col_entero = capitulos["X_col"].astype("int64")
x_mun_entero = capitulos["X_mun"].astype("int64")
xt_col_entero = np.int64(TOTALES["X_col"])
xt_mun_entero = np.int64(TOTALES["X_mun"])

with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    nrca_malo = x_col_entero / xt_mun_entero - (xt_col_entero * x_mun_entero) / (xt_mun_entero ** 2)

print("NRCA calculado con enteros (MAL):\n")
print(f"  Capitulos con NRCA positivo : {int((nrca_malo > 0).sum())} de {len(capitulos)}")
print(f"  Suma de todos los NRCA      : {nrca_malo.sum():+.6f}   (deberia ser cero)")
print()
print("  Los 97 capitulos salen positivos. Eso es imposible: significaria que Colombia")
print("  tiene ventaja comparativa en absolutamente todo, incluida la maquinaria")
print("  electrica que importa masivamente. Y la propiedad de conservacion no se cumple.")

In [ ]:
def nrca(x_pais, x_pais_total, x_mundo, x_mundo_total):
    '''Indice Normalizado de Ventaja Comparativa Revelada (Yu, Cai y Leung, 2009).

    Mide la distancia entre lo que el pais exporta del producto y lo que
    exportaria si tuviera un peso "neutral" acorde a su tamano en el comercio
    mundial. Positivo = ventaja; negativo = desventaja; suma cero por construccion.

    Los float() no son decorativos: el denominador eleva al cuadrado el comercio
    mundial total, que desborda un int64 y devuelve basura sin avisar.
    '''
    x_pais  = np.asarray(x_pais,  dtype=float)
    x_mundo = np.asarray(x_mundo, dtype=float)
    x_pais_total  = float(x_pais_total)
    x_mundo_total = float(x_mundo_total)

    observado = x_pais / x_mundo_total
    neutral   = (x_pais_total * x_mundo) / (x_mundo_total ** 2)

    return observado - neutral

In [ ]:
capitulos["NRCA"] = nrca(
    capitulos["X_col"], TOTALES["X_col"],
    capitulos["X_mun"], TOTALES["X_mun"],
)

suma = capitulos["NRCA"].sum()

print("NRCA calculado en punto flotante (BIEN):\n")
print(f"  Capitulos con NRCA positivo : {int((capitulos['NRCA'] > 0).sum())} de {len(capitulos)}")
print(f"  Capitulos con NRCA negativo : {int((capitulos['NRCA'] < 0).sum())}")
print(f"\n  CONTROL - suma de todos los NRCA: {suma:+.3e}")

assert abs(suma) < 1e-8, "La propiedad de conservacion no se cumple: revisa los tipos de dato"
print("  La propiedad de conservacion se cumple. El calculo es correcto.")

print(f"\n  Y coincide con Balassa en el diagnostico: los {int((capitulos['NRCA'] > 0).sum())} capitulos con NRCA")
print(f"  positivo son exactamente los {int((capitulos['RCA'] > 1).sum())} que tienen RCA > 1.")
print(f"  Coinciden capitulo por capitulo: {bool((( capitulos['NRCA'] > 0) == (capitulos['RCA'] > 1)).all())}")

**Para pensar.** El error no lanzó ninguna excepción. El cuaderno habría seguido corriendo, los gráficos se habrían dibujado, y la conclusión —"Colombia tiene ventaja comparativa en los 97 capítulos"— habría sido absurda pero perfectamente presentable.

Lo único que lo detectó fue la **propiedad de conservación**: un control que la propia teoría del índice regala y que cuesta una línea de código.

¿Cuántos de tus cálculos tienen un control así? ¿Y qué haces cuando la métrica que usas no trae ninguno?

## 4.6. Lo que el NRCA ve y Balassa no

Ambos índices están de acuerdo en **quién** tiene ventaja. Pero los ordenan de forma muy distinta, y esa diferencia es el aporte real del NRCA.

In [ ]:
comparacion = capitulos[["codigo", "producto", "X_col", "RCA", "NRCA"]].copy()
comparacion["puesto_RCA"]  = comparacion["RCA"].rank(ascending=False).astype(int)
comparacion["puesto_NRCA"] = comparacion["NRCA"].rank(ascending=False).astype(int)
comparacion["salto"] = comparacion["puesto_RCA"] - comparacion["puesto_NRCA"]

print(f"Correlacion de rangos entre RCA y NRCA (Spearman): "
      f"{capitulos['RCA'].corr(capitulos['NRCA'], method='spearman'):.4f}\n")

print("Los ocho primeros segun cada indice:\n")
izq = comparacion.nsmallest(8, "puesto_RCA")[["codigo", "RCA"]].reset_index(drop=True)
der = comparacion.nsmallest(8, "puesto_NRCA")[["codigo", "NRCA", "X_col"]].reset_index(drop=True)
lado_a_lado = pd.concat([izq.add_prefix("rca_"), der.add_prefix("nrca_")], axis=1)
print(lado_a_lado.to_string(index=False))

Balassa pone las **flores** en primer lugar; el NRCA pone los **combustibles minerales**.

Ninguno se equivoca: están midiendo cosas distintas.

- El **RCA** mide **intensidad**: qué tan desproporcionadamente importante es un producto dentro de la canasta del país. Colombia vende poca cantidad de flores en términos absolutos, pero para su tamaño vende muchísima.
- El **NRCA** mide **magnitud de la desviación**: cuántos dólares por encima de lo neutral exporta el país. Los combustibles mueven siete veces más dinero que las flores, así que su desviación absoluta es mayor aunque su intensidad relativa sea menor.

Para una decisión de inversión la distinción es material. Si la pregunta es *"¿dónde es Colombia inusualmente buena?"*, responde el RCA. Si es *"¿dónde se juega Colombia el mayor volumen de su ventaja?"*, responde el NRCA. La empresa del caso vive en el primer mundo; el Ministerio de Comercio, en el segundo.

## 4.7. Interpretación y umbrales

Rango simétrico, con **0 como estado neutro exacto** de competitividad. Valores positivos y **crecientes en el tiempo** son la señal más confiable, entre toda la familia RCA, de una ventaja comparativa genuina y estadísticamente robusta.

## 4.8. Impacto en el negocio y la decisión que habilita

El NRCA es, junto con el HHI, la combinación metodológica más usada entre 2024 y 2026 para decisiones de diversificación de mercados basadas en evidencia: salmón (García Juárez et al., 2026), maíz (Arbulú Ballesteros et al., 2026), banano andino (Corrales Otazú et al., 2025), calzado vietnamita (Hasan et al., 2024), textiles estadounidenses (Saki et al., 2019) y comercio bilateral Colombia-China (Morales Sánchez et al., 2026).

Volveremos sobre esa combinación en la sección 6, que es donde este cuaderno responde al caso.

## 4.9. De dónde se saca exactamente el dato

Igual que el RCA de Balassa, sobre exportaciones bilaterales y mundiales de Trade Map (ITC, 2025).

---
# 5. Métrica 11 — El Índice de Lafay: especialización aislada del ciclo

## 5.1. Qué es

A diferencia de los índices anteriores, que comparan el desempeño de un producto contra el **tamaño del mercado mundial**, el Índice de Lafay (1992) lo compara contra el **desempeño comercial estructural del propio país**: contra su propio promedio de balanza comercial en todos los bienes.

## 5.2. Para qué sirve y cómo se usa

Se usa específicamente para **productos primarios y agroindustriales**, donde el ciclo macroeconómico —inflación, recesión, choques climáticos— puede inflar o deprimir artificialmente el volumen de comercio de un año a otro, distorsionando cualquier índice basado en comparación mundial.

El Índice de Lafay "filtra" ese ruido cíclico, aislando si el producto realmente se comporta mejor o peor que el promedio estructural del propio país.

## 5.3. La fórmula

$$LFI^j_i = 100 \times \left( \frac{X^j_i - M^j_i}{X^j_i + M^j_i} - \frac{\sum_{j=1}^{n}(X^j_i - M^j_i)}{\sum_{j=1}^{n}(X^j_i + M^j_i)} \right) \times \frac{X^j_i + M^j_i}{\sum_{j=1}^{n}(X^j_i + M^j_i)}$$

## 5.4. Explicación matemática detallada

La fórmula parece intimidante, pero se descompone en **tres piezas reconocibles**:

**Pieza 1** — $\dfrac{X^j_i - M^j_i}{X^j_i + M^j_i}$ es exactamente el **IBCR del producto $j$**, el índice del Cuaderno 2. Mide si ese producto es superavitario o deficitario, en escala de −1 a +1.

**Pieza 2** — $\dfrac{\sum(X^j_i - M^j_i)}{\sum(X^j_i + M^j_i)}$ es el mismo cálculo pero agregado a toda la economía: el **"IBCR promedio nacional"**, el balance comercial estructural de fondo del país.

Restar la segunda de la primera responde: *"¿este producto tiene un balance comercial mejor o peor que el promedio del país?"*. Y ahí está el truco anticíclico: si una devaluación mejora la balanza de todos los productos a la vez, mejora también el promedio nacional, y la resta lo cancela.

**Pieza 3** — $\dfrac{X^j_i + M^j_i}{\sum(X^j_i + M^j_i)}$ es el peso del comercio de ese producto dentro del comercio total del país. Multiplicar por él evita que un producto marginal pese lo mismo en la conclusión que uno que representa una porción grande de la economía.

Multiplicar por 100 solo expresa el resultado en una escala más legible. Y, igual que el NRCA, **la suma de todos los LFI da cero**: otro control gratuito.

## 5.5. Ejemplo numérico paso a paso

In [ ]:
def lafay(x_pais, m_pais):
    '''Indice de Lafay (1992): especializacion aislada del ciclo macroeconomico.

    Compara el balance comercial de cada producto contra el balance estructural
    del propio pais, ponderado por el peso del producto en el comercio total.
    Positivo = el producto va mejor que el promedio del pais. Suma cero.
    '''
    x_pais = np.asarray(x_pais, dtype=float)
    m_pais = np.asarray(m_pais, dtype=float)

    comercio_producto = x_pais + m_pais
    comercio_total    = comercio_producto.sum()

    ibcr_producto = (x_pais - m_pais) / comercio_producto          # pieza 1
    ibcr_nacional = (x_pais - m_pais).sum() / comercio_total       # pieza 2
    peso_producto = comercio_producto / comercio_total             # pieza 3

    return 100 * (ibcr_producto - ibcr_nacional) * peso_producto

In [ ]:
# Un ejemplo minimo de tres productos, para ver las tres piezas por separado
ejemplo = pd.DataFrame({
    "producto": ["Cafe", "Maquinaria", "Textiles"],
    "X": [5_000.0, 200.0, 800.0],
    "M": [100.0, 3_000.0, 900.0],
})

comercio = ejemplo["X"] + ejemplo["M"]
ejemplo["IBCR_producto"] = (ejemplo["X"] - ejemplo["M"]) / comercio
ejemplo["IBCR_nacional"] = (ejemplo["X"] - ejemplo["M"]).sum() / comercio.sum()
ejemplo["peso"]          = comercio / comercio.sum()
ejemplo["LFI"]           = lafay(ejemplo["X"], ejemplo["M"])

print("Las tres piezas de la formula, una a una:\n")
print(ejemplo.to_string(index=False))
print(f"\n  Suma de los LFI: {ejemplo['LFI'].sum():+.2e}   (debe ser cero)")
print()
print("  El cafe tiene IBCR de +0,96 frente a un promedio nacional de +0,29:")
print("  va mucho mejor que el pais. La maquinaria, al reves. Los textiles estan")
print("  casi equilibrados y por eso su LFI es pequeno pese a no ser marginales.")

## 5.6. Cálculo sobre nuestros datos reales

In [ ]:
capitulos["LFI"] = lafay(capitulos["X_col"], capitulos["M_col"])

suma_lfi = capitulos["LFI"].sum()
print(f"CONTROL - suma de todos los LFI: {suma_lfi:+.2e}   (debe ser cero)")
assert abs(suma_lfi) < 1e-6, "La suma no cierra: revisa el calculo"
print("  El control pasa.\n")

print("Los cinco capitulos con MAYOR especializacion estructural:\n")
tabla = capitulos.nlargest(5, "LFI")[["codigo", "producto", "X_col", "M_col", "LFI"]].copy()
tabla["producto"] = tabla["producto"].str[:42]
print(tabla.to_string(index=False))

print("\n\nLos cinco con MAYOR dependencia estructural de importaciones:\n")
tabla = capitulos.nsmallest(5, "LFI")[["codigo", "producto", "X_col", "M_col", "LFI"]].copy()
tabla["producto"] = tabla["producto"].str[:42]
print(tabla.to_string(index=False))

In [ ]:
datos = pd.concat([capitulos.nlargest(7, "LFI"), capitulos.nsmallest(7, "LFI")]).sort_values("LFI")
etiquetas = [f"HS{c} · {p[:26]}" for c, p in zip(datos["codigo"], datos["producto"])]
colores = [NARANJA if c == "06" else (AZUL if v >= 0 else ROJO) for c, v in zip(datos["codigo"], datos["LFI"])]

fig, ax = plt.subplots(figsize=(9.5, 6.4))
barras = ax.barh(etiquetas, datos["LFI"], color=colores, height=0.68)

for barra, valor in zip(barras, datos["LFI"]):
    desplazamiento = 0.35 if valor >= 0 else -0.35
    alineacion = "left" if valor >= 0 else "right"
    ax.text(valor + desplazamiento, barra.get_y() + barra.get_height() / 2,
            f"{valor:+.2f}", va="center", ha=alineacion, fontsize=9, color=GRIS_TEXT)

ax.axvline(0, color=GRIS_MID, linewidth=1.4)
ax.set_xlabel("Índice de Lafay   ·   ponderado por el peso del capítulo en el comercio total",
              fontsize=10, color=GRIS_TEXT)
ax.set_title("Dónde Colombia va mejor —y peor— que su propio promedio comercial\n"
             "Índice de Lafay, siete capítulos más altos y siete más bajos, 2025",
             fontsize=13, color=TINTA, loc="left", pad=14)
ax.set_xlim(datos["LFI"].min() - 2.6, datos["LFI"].max() + 2.6)
ax.xaxis.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=8.5)

plt.figtext(0.01, -0.04,
            "En naranja HS06. La suma de todos los indices, incluidos los no mostrados, es cero.\n"
            "Fuente: elaboracion propia con datos de Trade Map (ITC, 2025).",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

## 5.7. Interpretación y umbrales

Un LFI positivo indica que el producto tiene un balance comercial **mejor que el promedio estructural del país**, ponderado por su importancia. Negativo, lo contrario. La suma sobre todos los productos es cero por construcción, así que el índice es intrínsecamente **relativo**: mide reparto, no nivel.

Fíjate en un detalle revelador: las flores tienen el RCA más alto de Colombia (43,6) pero apenas el cuarto LFI. La razón es la **ponderación por peso**: las flores son solo el 4,9 % de las exportaciones colombianas, así que por muy especializado que esté el país, su contribución al balance estructural es modesta. El RCA no pondera; el Lafay sí.

Sumi et al. (2025) aplican el índice a Bangladesh dentro de la SAARC y encuentran que cuatro de los cinco productos con mayor especialización estructural sostenida provienen del sector de confecciones.

## 5.8. Impacto en el negocio y la decisión que habilita

Para un ministerio que diseña política de sustitución de importaciones, los LFI más negativos son la lista de objetivos: los capítulos donde el país sangra divisas por encima de su patrón normal.

Para una empresa, el LFI del propio sector responde a una pregunta que el RCA no toca: *"¿mi industria aporta o resta al balance externo del país?"*. Esa respuesta determina, en la práctica, si la industria tendrá acceso a incentivos, líneas de crédito preferentes o atención política.

## 5.9. De dónde se saca exactamente el dato

Exportaciones e importaciones del país por producto, de Trade Map (ITC, 2025). A diferencia del resto de la familia, **no requiere datos del comercio mundial**: todo el cálculo es interno al país. Esa autosuficiencia es parte de su atractivo cuando los datos mundiales son poco fiables o llegan con retraso.

---
# 6. La familia completa, lado a lado

Cinco índices sobre los mismos datos. Aquí se ve por qué existen los cinco.

In [ ]:
FAMILIA = pd.DataFrame([
    ("Balassa (RCA)", 1965, "Cociente", "0 a infinito", "1",
     "Diagnostico rapido, comunicacion", "Ignora importaciones; asimetrico"),
    ("Laursen (RSCA)", 2015, "Cociente reescalado", "-1 a +1", "0",
     "Regresiones y pruebas estadisticas", "No cambia el orden, solo la escala"),
    ("Vollrath (RC)", 1991, "Diferencia de logaritmos", "simetrico en 0", "0",
     "Detectar maquila y dependencia de insumos", "Indefinido si algun flujo es cero"),
    ("Yu, Cai y Leung (NRCA)", 2009, "Diferencia", "simetrico en 0", "0",
     "Paneles econometricos, comparacion entre paises", "Pondera por tamano: favorece volumen"),
    ("Lafay (LFI)", 1992, "Diferencia ponderada", "suma cero", "0",
     "Productos primarios; aislar el ciclo", "Relativo al propio pais, no al mundo"),
], columns=["Indice", "Anio", "Construccion", "Rango", "Umbral", "Para que sirve", "Limitacion"])

print(FAMILIA.to_string(index=False))

In [ ]:
RESUMEN = capitulos[["codigo", "producto", "X_col", "RCA", "RSCA", "RXA", "RMA", "RC", "NRCA", "LFI"]].copy()

destacados = ["06", "09", "27", "71", "08", "85", "84", "87"]
tabla = RESUMEN[RESUMEN["codigo"].isin(destacados)].copy()
tabla["producto"] = tabla["producto"].str[:30]
tabla = tabla.sort_values("RCA", ascending=False)

print("Los cinco indices sobre ocho capitulos representativos:\n")
print(tabla.to_string(index=False))

In [ ]:
# ¿Que tanto coinciden entre si? Correlacion de rangos, que es lo que importa
indices = ["RCA", "RSCA", "RC", "NRCA", "LFI"]
correlaciones = capitulos[indices].corr(method="spearman")

print("Correlacion de rangos (Spearman) entre los cinco indices:\n")
print(correlaciones.round(3).to_string())
print()
print("  RCA y RSCA correlacionan 1,000: son el mismo orden en distinta escala.")
print("  RCA y NRCA correlacionan mucho menos: ordenan por criterios distintos.")
print("  Esa es la razon de calcular mas de uno. Si todos dieran lo mismo, bastaria uno.")

---
# 7. El cruce que responde al caso: HHI × NRCA

El documento base señala la combinación **HHI + NRCA** como la metodología más usada en la investigación aplicada de 2024-2026 para decisiones de diversificación de mercados. La lógica es que cada índice cubre el punto ciego del otro:

- El **NRCA** dice si hay **ventaja competitiva real** en un producto, pero no dice nada del riesgo de a quién se le vende.
- El **HHI** dice si las ventas están peligrosamente concentradas, pero no dice nada de si el producto es competitivo.

Cruzados, generan cuatro cuadrantes con lecturas de negocio muy distintas. Trade Map publica, para cada capítulo exportado por Colombia, la concentración de sus mercados de destino: eso nos da el eje del HHI.

In [ ]:
# La vista de indicadores trae la concentracion de mercados de destino por capitulo.
# Sus nombres de columna reales estan en la segunda fila, no en la primera.
bruto = max(pd.read_html(ARCHIVOS["co_exp_ind"]), key=lambda t: t.shape[0])
nombres = [str(x).strip() for x in bruto.iloc[1].tolist()]

indicadores = bruto.iloc[2:].copy()
indicadores.columns = nombres
indicadores = indicadores.loc[:, ~indicadores.columns.duplicated()]
indicadores = indicadores.rename(columns={"Code": "codigo"})
indicadores["codigo"] = limpiar_codigo(indicadores["codigo"])

columna_hhi = [c for c in indicadores.columns if c.startswith("Concentration")][0]
indicadores["hhi_destinos"] = a_numero(indicadores[columna_hhi])

print(f"Columna encontrada: '{columna_hhi}'\n")

cruce = capitulos.merge(indicadores[["codigo", "hhi_destinos"]], on="codigo", how="left")
cruce = cruce.dropna(subset=["hhi_destinos"])
print(f"Capitulos con dato de concentracion de destinos: {len(cruce)} de {len(capitulos)}")
print(f"\n  HS06 (flores): NRCA = {cruce.loc[cruce['codigo'] == '06', 'NRCA'].iloc[0]:+.6f}   "
      f"HHI de destinos = {cruce.loc[cruce['codigo'] == '06', 'hhi_destinos'].iloc[0]:.4f}")

In [ ]:
grandes = cruce.nlargest(28, "X_col")

corte_nrca = 0.0
corte_hhi  = grandes["hhi_destinos"].median()

fig, ax = plt.subplots(figsize=(9.8, 6.6))

ax.axhline(corte_hhi, color=GRIS_MID, linewidth=1.2, zorder=1)
ax.axvline(corte_nrca, color=GRIS_MID, linewidth=1.2, zorder=1)

tamanos = 70 + 620 * (grandes["X_col"] / grandes["X_col"].max())
colores = [NARANJA if c == "06" else (AZUL if n > 0 else ROJO)
           for c, n in zip(grandes["codigo"], grandes["NRCA"])]

ax.scatter(grandes["NRCA"], grandes["hhi_destinos"], s=tamanos, c=colores,
           alpha=0.72, edgecolors="white", linewidths=1.2, zorder=3)

for _, fila in grandes.iterrows():
    ax.annotate(f"HS{fila['codigo']}", (fila["NRCA"], fila["hhi_destinos"]),
                textcoords="offset points", xytext=(0, -3.5),
                ha="center", fontsize=7.5, color=TINTA, zorder=4)

ax.set_xlabel("NRCA   ·   ventaja comparativa revelada normalizada", fontsize=10, color=GRIS_TEXT)
ax.set_ylabel("HHI de los mercados de destino   ·   riesgo de concentración", fontsize=10, color=GRIS_TEXT)
ax.set_title("Competitivo y diversificado: el cuadrante que buscan las agencias de exportación\n"
             "Cruce NRCA × HHI, 28 capítulos con más exportación colombiana, 2025",
             fontsize=13, color=TINTA, loc="left", pad=16)

ax.text(0.985, 0.975, "VENTAJA, pero concentrado\n(diversificar destinos)",
        transform=ax.transAxes, ha="right", va="top", fontsize=8.5, color=GRIS_EJE)
ax.text(0.985, 0.03, "VENTAJA y diversificado\n(el objetivo)",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=8.5, color=GRIS_EJE)
ax.text(0.015, 0.975, "sin ventaja y concentrado\n(el peor cuadrante)",
        transform=ax.transAxes, ha="left", va="top", fontsize=8.5, color=GRIS_EJE)

ax.grid(True, color=REJILLA, linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right"]:
    ax.spines[lado].set_visible(False)
for lado in ["bottom", "left"]:
    ax.spines[lado].set_color(GRIS_EJE)
ax.tick_params(colors=GRIS_EJE, labelsize=9)

plt.figtext(0.01, -0.03,
            "El tamano del circulo es el valor exportado. En naranja HS06, el capitulo del caso.\n"
            "Fuente: elaboracion propia con datos de Trade Map (ITC, 2025).",
            fontsize=8, color=GRIS_EJE, ha="left")
plt.tight_layout()
plt.show()

## 7.1. La respuesta al caso

Ya podemos responder la pregunta con la que abrió el cuaderno.

**¿La posición colombiana en flores es ventaja comparativa real o volumen heredado?**

Es ventaja real, y los cinco índices coinciden por caminos independientes:

| Índice | Valor en HS06 | Qué certifica |
|---|---|---|
| **RCA de Balassa** | 43,6 | La especialización más alta de toda la economía colombiana |
| **RSCA de Laursen** | +0,955 | A un paso del máximo teórico de +1 |
| **RC de Vollrath** | +3,87 | La ventaja **no** depende de insumos importados: no es maquila |
| **NRCA** | positivo y en el tercer puesto | Robusto bajo el índice más exigente de la familia |
| **Lafay** | +2,31 | El sector aporta al balance estructural del país |

El **RC de Vollrath** es el que más aporta aquí. Con un RMA de apenas 1,04, Colombia casi no importa flores: exporta lo que produce, no lo que ensambla. Esa es la diferencia entre una ventaja que se sostiene y una que depende de una cadena de suministro externa vulnerable a devaluaciones o aranceles.

**Y la decisión que habilita.** Sumando el Cuaderno 3 y este:

1. La ventaja competitiva en flores es real, estructural y creciente (el RCA subió de 34,3 a 43,6 en cinco años).
2. Pero el mercado coreano ya está saturado para Colombia: 86,65 % de cuota. **Ahí no hay espacio de crecimiento por sustitución de competidores**; solo por crecimiento del propio mercado, que viene creciendo al 15 % anual.
3. Y el HHI por destino apenas se mueve: 4,8 destinos equivalentes, con Estados Unidos anclado en el 41 %.

La recomendación al comité no es *"entrar a Corea"* —ya están— ni *"crecer en Corea"* —hay poco que tomar—. Es **usar la ventaja demostrada en flores para abrir mercados donde Colombia todavía no es el proveedor dominante**, porque la ventaja competitiva viaja y la concentración de clientes es el riesgo que sigue sin resolverse.

Esa conclusión no se podía formular con las métricas del Cuaderno 2. Y no habría sido correcta sin haber descubierto, en el Cuaderno 3, que la premisa original del caso era falsa.

In [ ]:
# Guardamos la tabla completa de la familia VCR
RESUMEN_FINAL = capitulos[["codigo", "producto", "X_col", "M_col", "X_mun", "M_mun",
                           "RCA", "RSCA", "RXA", "RMA", "RTA", "RC", "NRCA", "LFI"]]

salidas = {
    "familia_vcr_colombia_hs2_2025.csv": RESUMEN_FINAL,
    "familia_vcr_comparativa.csv":       FAMILIA,
    "cruce_nrca_hhi.csv":                cruce[["codigo", "producto", "X_col", "NRCA", "hhi_destinos"]],
    "rca_serie_2021_2025.csv":           serie_rca,
}

for nombre, tabla in salidas.items():
    ruta = os.path.join(CARPETA_SALIDA, nombre)
    tabla.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"  guardado: {ruta}  ({len(tabla)} filas)")

# Si estas en Colab y quieres bajarlos:
# from google.colab import files
# for nombre in salidas: files.download(os.path.join(CARPETA_SALIDA, nombre))

---
# 8. Los límites de esta familia y el puente al Documento 3

Los cinco índices comparten cuatro limitaciones que ninguna corrección interna resuelve:

| Límite | Qué significa |
|---|---|
| **Son estáticos** | Retratan un año. Calculamos la serie del RCA, pero el índice en sí no modela la dinámica |
| **No explican el cambio** | Sabemos que el RCA de flores subió de 34,3 a 43,6. Ninguno dice **por qué**: ¿Colombia mejoró, o los competidores empeoraron? |
| **No predicen** | Describen lo revelado por el comportamiento pasado. No estiman qué pasaría si cambiara un arancel o un tipo de cambio |
| **No seleccionan mercados** | Dicen en qué producto hay ventaja, no a qué país conviene venderlo |

**Lo que viene.** El Documento 3 —nivel avanzado— responde exactamente esas cuatro preguntas:

- **Constant Market Share (CMS)** — descompone el cambio en las exportaciones para separar cuánto se debe a que el mercado mundial creció, cuánto a estar en los productos correctos, y cuánto a haber ganado competitividad real. Es la respuesta directa al segundo límite.
- **Econometría gravitacional (PPML)** — modela el comercio bilateral en función de tamaño, distancia y barreras, y sí permite estimar efectos de política.
- **IMSFEOG** — la técnica multicriterio de Baena-Rojas y Cano (2026), con sus seis factores y dieciocho variables, para seleccionar mercados de forma sistemática.

Ahí es donde el análisis pasa de describir a decidir.

---
# 9. Glosario mínimo

**Ventaja comparativa revelada.** Especialización que se **infiere** del comportamiento comercial observado, no de un supuesto teórico sobre productividad o dotación de factores. De ahí "revelada".

**Punto neutro / ventaja comparativa neutral (CAN).** La participación que un país tendría en un producto si exportara exactamente en proporción a su peso general en el comercio mundial. Es el punto de referencia del NRCA.

**Asimetría estadística.** Defecto del RCA de Balassa: el rango de desventaja está acotado entre 0 y 1 mientras que el de ventaja va de 1 a infinito. Impide su uso directo en regresiones sin distorsión.

**Propiedad de conservación.** Característica del NRCA y del Lafay: la suma sobre todos los productos da exactamente cero. Sirve como control de que el cálculo está bien hecho.

**Desbordamiento de enteros (*integer overflow*).** Error silencioso que ocurre cuando un resultado excede el valor máximo del tipo de dato. NumPy no lanza excepción: devuelve un número equivocado. Ver sección 4.5.

**Maquila / ensamblaje.** Modelo en que un país importa componentes, los ensambla y reexporta. Produce un RCA alto que no refleja capacidad productiva propia. Es lo que detecta el RC de Vollrath.

**Correlación de rangos (Spearman).** Mide si dos índices ordenan los elementos igual, sin importar la escala. Es la forma correcta de comparar índices con rangos distintos.

**RXA / RMA / RTA / RC.** Ventaja Relativa de Exportación, de Importación, Comercial Relativa (su diferencia) y Competitividad Revelada (diferencia de sus logaritmos). La familia Vollrath.

---
# 10. Ejercicios propuestos

**Ejercicio 1 — El linaje en una frase.** Escribe, para cada uno de los cinco índices, una sola frase que empiece con "Existe porque el anterior...". Si no puedes completarla, vuelve a la sección correspondiente: no has entendido para qué sirve.

**Ejercicio 2 — Reproducir el desborde.** Vuelve a la sección 4.5 y calcula el NRCA en `int64` pero con los valores expresados en **millones** en vez de miles de dólares. ¿Se desborda igual? ¿A partir de qué magnitud deja de fallar? Explica por qué cambiar la unidad puede ocultar un error sin arreglarlo.

**Ejercicio 3 — Los capítulos desmentidos.** La sección 3.7 encontró cuatro capítulos con RCA > 1 y RC < 0. Escoge uno, busca qué productos concretos contiene a 4 dígitos, y argumenta si el diagnóstico de "ventaja aparente por insumos importados" tiene sentido económico para ese sector colombiano.

**Ejercicio 4 — RCA contra NRCA.** Calcula, para los 97 capítulos, la diferencia entre el puesto en el *ranking* del RCA y el del NRCA. ¿Qué capítulo es el que más sube y cuál el que más baja? Explica ambos casos con la distinción intensidad/magnitud de la sección 4.6.

**Ejercicio 5 — Lafay sin ponderar.** Recalcula el índice de Lafay eliminando la tercera pieza (el peso del producto). ¿Cambia el orden de los cinco primeros? ¿Qué capítulos aparecen ahora que antes no estaban, y por qué el propio Lafay decidió ponderar?

**Ejercicio 6 — Otro país.** Descarga de Trade Map la canasta exportadora e importadora de **Ecuador** o **Kenia** —los dos competidores florícolas de Colombia— y calcula los cinco índices para su capítulo 06. ¿Quién tiene la ventaja comparativa más sólida? ¿Coinciden los cinco índices en la respuesta?

**Ejercicio 7 — La serie completa.** El objeto `serie_rca` tiene el RCA de los cinco años. Identifica los tres capítulos donde más creció y los tres donde más cayó entre 2021 y 2025. ¿Alguno cruzó el umbral de 1 en alguna dirección? Ese cruce es una noticia de política comercial.

**Ejercicio 8 — El cuadrante vacío.** Mira el gráfico de la sección 7. ¿Hay capítulos en el cuadrante "ventaja y diversificado"? Si la respuesta es que muy pocos, ¿qué dice eso de la estrategia exportadora colombiana en su conjunto?

---
# Referencias

Arbulú Ballesteros, M., et al. (2026). Competitividad y concentración de mercados en las exportaciones mundiales de maíz. *Revista de Economía Agrícola*.

Baena-Rojas, J. J., & Cano, J. A. (2026). International market selection for exports of goods: A data analysis technique for organizational decision-making. *Global Business Review*. https://doi.org/10.1177/09721509261464305

Balassa, B. (1965). Trade liberalisation and "revealed" comparative advantage. *The Manchester School, 33*(2), 99–123.

Bayav, İ., & Engindeniz, S. (2021). Revealed comparative advantage of Turkish peach exports. *Journal of Agricultural Sciences*.

Corrales Otazú, J., et al. (2025). Ventaja comparativa revelada y concentración de mercados en las exportaciones de banano de la Comunidad Andina. *Revista Iberoamericana de Comercio Exterior*.

Danna-Buitrago, J. P., & Gnabo, J.-Y. (2021). A new class of revealed comparative advantage indexes. *Journal of Economic Studies*.

de Benedictis, L., & Tamberi, M. (2001). *A note on the Balassa index of revealed comparative advantage* (Working Paper). Università Politecnica delle Marche.

French, S. (2017). Revealed comparative advantage: What is it good for? *Journal of International Economics, 106*, 83–103.

García Juárez, R., et al. (2026). Índice normalizado de ventaja comparativa revelada y concentración de mercados en las exportaciones de salmón. *Aquaculture Economics & Management*.

Gül, M., et al. (2024). Revealed comparative advantage in agriculture: A twelve-country comparison. *Agricultural Economics Review*.

Harris, C. R., et al. (2020). Array programming with NumPy. *Nature, 585*, 357–362. https://doi.org/10.1038/s41586-020-2649-2

Hasan, M., et al. (2024). Competitiveness of Vietnamese footwear exports: An NRCA approach. *Journal of Asian Business Studies*.

Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55

International Trade Centre. (2025). *Trade Map: Trade statistics for international business development*. https://www.trademap.org

Lafay, G. (1992). The measurement of revealed comparative advantages. En M. G. Dagenais & P.-A. Muet (Eds.), *International trade modelling* (pp. 209–234). Chapman & Hall.

Laursen, K. (2015). Revealed comparative advantage and the alternatives as measures of international specialization. *Eurasian Business Review, 5*(1), 99–115.

Liu, B., et al. (2019). On the asymmetry of the revealed comparative advantage index. *Applied Economics Letters*.

Maqbool, M. S., et al. (2021). Revealed competitiveness of ASEAN electronics exports. *Asian Economic Journal*.

McKinney, W. (2010). Data structures for statistical computing in Python. En S. van der Walt & J. Millman (Eds.), *Proceedings of the 9th Python in Science Conference* (pp. 56–61). https://doi.org/10.25080/Majora-92bf1922-00a

Mohd Yusoff, R., et al. (2022). Revealed comparative advantage in the global halal meat trade. *Journal of Islamic Marketing*.

Morales Sánchez, A., et al. (2026). Ventaja comparativa revelada en el comercio bilateral Colombia-China. *Revista de Economía Internacional*.

Sałamaga, M. (2015). The application of the revealed comparative advantage index in the analysis of foreign trade. *Statistics in Transition*.

Saki, Z., et al. (2019). Revealed comparative advantage of the US textile and apparel industry. *Journal of the Textile Institute*.

Serie de recursos — Inteligencia en Negocios Globales. (2026). *Métricas de comercio exterior e inteligencia de negocios globales. Documento 2 de 3 — Nivel intermedio: concentración, comercio intraindustrial y ventaja comparativa revelada*. Universidad EAN.

Sumi, R., et al. (2025). Structural specialization of Bangladesh within SAARC: A Lafay index approach. *South Asian Economic Journal*.

Vollrath, T. L. (1991). A theoretical evaluation of alternative trade intensity measures of revealed comparative advantage. *Weltwirtschaftliches Archiv, 127*(2), 265–280.

Yu, R., Cai, J., & Leung, P. (2009). The normalized revealed comparative advantage index. *The Annals of Regional Science, 43*(1), 267–282.